In [ ]:
"""
======================================================================
Image Captioning with Vision-Language Models (CS6350 - TPA19)
======================================================================
End-to-end pipeline: fine-tune a pretrained BLIP model on MS-COCO
captions, evaluate with BLEU / CIDEr / METEOR / ROUGE-L, and run
qualitative + zero-shot analysis.

Run stages independently via --stage, or `all` for the full run.
Designed for a single GPU (Colab/Kaggle T4 or better).

    python image_captioning_pipeline.py --stage prepare_data
    python image_captioning_pipeline.py --stage train
    python image_captioning_pipeline.py --stage evaluate
    python image_captioning_pipeline.py --stage qualitative
    python image_captioning_pipeline.py --stage zero_shot

Install (Colab):
    pip install -q transformers datasets accelerate pillow \
        pycocotools pycocoevalcap nltk rouge-score sacrebleu \
        matplotlib torch torchvision
======================================================================
"""

import os
import json
import random
import argparse
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    get_linear_schedule_with_warmup,
)


In [ ]:
# ----------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------
CONFIG = {
    "model_name": "Salesforce/blip-image-captioning-base",  # pretrained checkpoint
    "data_root": "./coco_data",          # where images/annotations live
    "checkpoint_dir": "./checkpoints",
    "output_dir": "./outputs",
 
    # --- Source annotation files -----------------------------------
    # train2017 is the raw file we split ourselves (see split_train_val).
    "train_ann": "annotations/captions_train2017.json",
    "train_img_dir": "train2017",
 
    # These two are *generated* by split_train_val() from train_ann.
    "train_split_ann": "annotations/captions_train_split.json",
    "val_split_ann": "annotations/captions_val_split.json",
    "val_split_ratio": 0.1,      # fraction of train2017 images held out for tuning
    "split_seed": 42,
 
    # val2017 is NEVER used for training or tuning — held-out test set only.
    "test_ann": "annotations/captions_val2017.json",
    "test_img_dir": "val2017",
    "max_test_samples": 5000,    # full val2017
 
    "max_train_samples": 100000,     # subsample the train split for feasible training time
    "max_tune_val_samples": 500,    # subsample the val split, for speed during Optuna trials
 
    "batch_size": 16,
    "num_epochs": 1,
    "lr": 5e-5,
    "warmup_ratio": 0.05,
    "max_caption_len": 40,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
 
    # --- Optuna ------------------------------------------------------
    "n_trials": 16,          # number of Optuna trials
    "tune_max_steps": 200,   # cap steps per trial so search stays fast
    "tune_epochs": 1,
}
 
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])



# ----------------------------------------------------------------------
# 1. DATASET
# ----------------------------------------------------------------------
class CocoCaptionDataset(Dataset):
    """
    Wraps a COCO-format captions_*.json. Each item = (image, one caption).
    COCO has ~5 captions/image; we flatten to (image_id, caption) pairs
    so each training step sees a single image-caption target.
    """
 
    def __init__(self, ann_path, img_dir, processor, max_samples=None):
        with open(ann_path, "r") as f:
            coco = json.load(f)
 
        id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
        self.samples = [
            {
                "image_path": os.path.join(img_dir, id_to_file[a["image_id"]]),
                "caption": a["caption"].strip(),
            }
            for a in coco["annotations"]
            if a["image_id"] in id_to_file
        ]
 
        random.shuffle(self.samples)
        if max_samples:
            self.samples = self.samples[:max_samples]
 
        self.processor = processor
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        encoding = self.processor(
            images=image,
            text=item["caption"],
            padding="max_length",
            truncation=True,
            max_length=CONFIG["max_caption_len"],
            return_tensors="pt",
        )
        # squeeze batch dim added by processor
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding["input_ids"][0] = 30522            # BLIP's [DEC] start token, same as generate()
        labels = encoding["input_ids"].clone()
        labels[encoding["attention_mask"] == 0] = -100   # don't train on padding
        encoding["labels"] = labels
        return encoding
 
 
def split_train_val():
    """
    Splits captions_train2017.json into a train split and a val split,
    by image_id (so all ~5 captions of one image land in the same
    split — no leakage across splits). The val split is used ONLY for
    Optuna hyperparameter tuning, never for final reporting.
 
    Writes:
        coco_data/annotations/captions_train_split.json
        coco_data/annotations/captions_val_split.json
    """
    root = Path(CONFIG["data_root"])
    train_ann_path = root / CONFIG["train_ann"]
    with open(train_ann_path, "r") as f:
        coco = json.load(f)
 
    image_ids = [img["id"] for img in coco["images"]]
    rng = random.Random(CONFIG["split_seed"])
    rng.shuffle(image_ids)
 
    n_val = int(len(image_ids) * CONFIG["val_split_ratio"])
    val_ids = set(image_ids[:n_val])
    train_ids = set(image_ids[n_val:])
 
    train_images = [img for img in coco["images"] if img["id"] in train_ids]
    val_images = [img for img in coco["images"] if img["id"] in val_ids]
    train_anns = [a for a in coco["annotations"] if a["image_id"] in train_ids]
    val_anns = [a for a in coco["annotations"] if a["image_id"] in val_ids]
 
    train_split = {"images": train_images, "annotations": train_anns}
    val_split = {"images": val_images, "annotations": val_anns}
 
    out_dir = root / "annotations"
    out_dir.mkdir(parents=True, exist_ok=True)
 
    train_split_path = out_dir / Path(CONFIG["train_split_ann"]).name
    val_split_path = out_dir / Path(CONFIG["val_split_ann"]).name
 
    with open(train_split_path, "w") as f:
        json.dump(train_split, f)
    with open(val_split_path, "w") as f:
        json.dump(val_split, f)
 
    print(f"Train split: {len(train_images)} images, {len(train_anns)} captions -> {train_split_path}")
    print(f"Val split (tuning only): {len(val_images)} images, {len(val_anns)} captions -> {val_split_path}")
    return train_split_path, val_split_path
 
 
def prepare_data():
    """
    Expects COCO already downloaded under CONFIG['data_root']:
        coco_data/annotations/captions_train2017.json
        coco_data/annotations/captions_val2017.json
        coco_data/train2017/*.jpg
        coco_data/val2017/*.jpg
 
    Download commands (run once, ~19GB total for full train2017):
        wget http://images.cocodataset.org/zips/train2017.zip
        wget http://images.cocodataset.org/zips/val2017.zip
        wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
        unzip -q train2017.zip -d coco_data/
        unzip -q val2017.zip -d coco_data/
        unzip -q annotations_trainval2017.zip -d coco_data/
 
    For a lighter class project, use only a subsample (max_train_samples)
    or switch to Flickr30k (HuggingFace: `nlphuji/flickr30k`).
 
    Also generates the internal train/val split (from train2017) used
    for hyperparameter tuning, if it doesn't exist yet. val2017 itself
    is left untouched — it is reserved as the final test set.
    """
    root = Path(CONFIG["data_root"])
    train_ann = root / CONFIG["train_ann"]
    test_ann = root / CONFIG["test_ann"]
    assert train_ann.exists() and test_ann.exists(), (
        "COCO annotations not found. Download COCO 2017 first — see the "
        "docstring in prepare_data() for the exact commands."
    )
    print("Data check passed. Train/test annotations found.")
 
    train_split_path = root / CONFIG["train_split_ann"]
    val_split_path = root / CONFIG["val_split_ann"]
    if train_split_path.exists() and val_split_path.exists():
        print("Train/val split already exists — skipping split_train_val().")
    else:
        print("Creating train/val split from train2017 for hyperparameter tuning...")
        split_train_val()

In [ ]:
# ----------------------------------------------------------------------
# 2. TRAINING
# ----------------------------------------------------------------------
def build_model_and_data(hparams, max_train_samples=None, max_val_samples=None):
    """Shared helper: loads a fresh model/processor + train & val datasets."""
    device = CONFIG["device"]
    processor = BlipProcessor.from_pretrained(CONFIG["model_name"])
    model = BlipForConditionalGeneration.from_pretrained(CONFIG["model_name"]).to(device)
 
    root = Path(CONFIG["data_root"])
    train_ds = CocoCaptionDataset(
        root / CONFIG["train_split_ann"],
        root / CONFIG["train_img_dir"],
        processor,
        max_samples=max_train_samples if max_train_samples is not None else CONFIG["max_train_samples"],
    )
    val_ds = CocoCaptionDataset(
        root / CONFIG["val_split_ann"],
        root / CONFIG["train_img_dir"],
        processor,
        max_samples=max_val_samples if max_val_samples is not None else CONFIG["max_tune_val_samples"],
    )
    return model, processor, train_ds, val_ds
 
 
def run_validation_loss(model, val_loader, device):
    """Average loss over the (tuning-only) val split — used as the Optuna objective."""
    model.eval()
    total_loss, n_batches = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            total_loss += outputs.loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)
 
 
def tune_hyperparameters(n_trials=None):
    """
    Optuna search over lr / batch_size / warmup_ratio, using the
    train2017-derived VAL split (never val2017) as the objective signal.
    Each trial trains for a capped number of steps (tune_max_steps) to
    keep the search tractable, then measures validation loss.
 
    Saves the best hyperparameters to outputs/best_hparams.json, which
    train() will pick up automatically if present.
    """
    import optuna
 
    n_trials = n_trials or CONFIG["n_trials"]
    device = CONFIG["device"]
 
    def objective(trial):
        lr = trial.suggest_float("lr", 1e-6, 1e-3, log=True)
        # 32 dropped from the search space — it OOMs on this GPU even
        # outside of tuning, so it's never a usable choice anyway.
        batch_size = trial.suggest_categorical("batch_size", [8, 16])
        warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.2)

        model = optimizer = scheduler = train_loader = val_loader = None
        try:
            model, processor, train_ds, val_ds = build_model_and_data(
                hparams={"lr": lr, "batch_size": batch_size, "warmup_ratio": warmup_ratio}
                )
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

            optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
            total_steps = min(len(train_loader) * CONFIG["tune_epochs"], CONFIG["tune_max_steps"])
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(warmup_ratio * total_steps),
                num_training_steps=max(total_steps, 1),
            )

            scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
            model.train()
            for step, batch in enumerate(train_loader):
                if step >= CONFIG["tune_max_steps"]:
                    break
                batch = {k: v.to(device) for k, v in batch.items()}
                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                    outputs = model(
                        pixel_values=batch["pixel_values"],
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                        labels=batch["labels"],
                    )
                    loss = outputs.loss
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            val_loss = run_validation_loss(model, val_loader, device)
            trial.report(val_loss, step=0)
            return val_loss

        except torch.cuda.OutOfMemoryError:
            # Don't kill the whole study over one bad (lr, batch_size)
            # combination — just mark this trial as pruned and move on.
            print(f"Trial {trial.number} OOM'd (batch_size={batch_size}, lr={lr:.2e}) — pruning.")
            raise optuna.exceptions.TrialPruned()

        finally:
            # Every reference that can hold GPU tensors alive across
            # trials gets dropped explicitly — relying on Python's
            # garbage collector alone (especially after an exception,
            # which keeps the traceback and its locals alive) is what
            # causes memory to creep up trial-over-trial until OOM.
            del model, optimizer, scheduler, train_loader, val_loader
            if device == "cuda":
                torch.cuda.empty_cache()
 
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
 
    print("Best hyperparameters:", study.best_params)
    print("Best val loss:", study.best_value)
 
    out_dir = Path(CONFIG["output_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "blip_best_hparams.json", "w") as f:
        json.dump(study.best_params, f, indent=2)
    print(f"Best hyperparameters saved to {out_dir / 'best_hparams.json'}")
 
    return study.best_params
 
 
import time

def train(hparams=None, run_name=None, num_epochs=None):
    """
    Fine-tunes BLIP on the TRAIN split (captions_train_split.json).
    If hparams is None, tries to load outputs/best_hparams.json (written
    by tune_hyperparameters()); otherwise falls back to CONFIG defaults.

    run_name: identifies this training run so checkpoints don't collide
    with other runs. Defaults to a timestamp if not given, so every call
    is automatically unique. Checkpoints save to:
        checkpoints/{run_name}/epoch{epoch}
    num_epochs: override CONFIG["num_epochs"] for this run specifically
    (e.g. train(run_name="1epoch_run", num_epochs=1) without touching CONFIG).
    """
    if run_name is None:
        run_name = time.strftime("run_%Y%m%d_%H%M%S")

    n_epochs = num_epochs if num_epochs is not None else CONFIG["num_epochs"]

    out_dir = Path(CONFIG["output_dir"])
    best_hparams_path = out_dir / "blip_best_hparams.json"
    if hparams is None and best_hparams_path.exists():
        with open(best_hparams_path) as f:
            hparams = json.load(f)
        print(f"Loaded tuned hyperparameters from {best_hparams_path}: {hparams}")
    hparams = hparams or {}

    lr = hparams.get("lr", CONFIG["lr"])
    batch_size = hparams.get("batch_size", CONFIG["batch_size"])
    warmup_ratio = hparams.get("warmup_ratio", CONFIG["warmup_ratio"])

    print(f"Run name: {run_name}  |  epochs: {n_epochs}")
    print("Loading model and processor...")
    device = CONFIG["device"]
    processor = BlipProcessor.from_pretrained(CONFIG["model_name"])
    model = BlipForConditionalGeneration.from_pretrained(CONFIG["model_name"]).to(device)

    root = Path(CONFIG["data_root"])
    train_ds = CocoCaptionDataset(
        root / CONFIG["train_split_ann"],
        root / CONFIG["train_img_dir"],
        processor,
        max_samples=CONFIG["max_train_samples"],
    )
    print(f"Dataset Loaded, {len(train_ds)}")
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=0
    )
    print(f"Starting training: {len(train_loader)} batches/epoch "
          f"(lr={lr}, batch_size={batch_size}, warmup_ratio={warmup_ratio})")
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * n_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(warmup_ratio * total_steps),
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
    model.train()

    run_ckpt_dir = Path(CONFIG["checkpoint_dir"]) / run_name

    for epoch in range(n_epochs):
        print(f"--- Epoch {epoch} ---")
        running_loss = 0.0
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                outputs = model(
                    pixel_values=batch["pixel_values"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                )
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()
            print(f"epoch {epoch} step {step}/{len(train_loader)} "
                  f"loss {running_loss / (step + 1):.4f}")

        ckpt_path = run_ckpt_dir / f"epoch{epoch}"
        ckpt_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ckpt_path)
        processor.save_pretrained(ckpt_path)
        print(f"Saved checkpoint: {ckpt_path}")

    val_ds = CocoCaptionDataset(
        root / CONFIG["val_split_ann"],
        root / CONFIG["train_img_dir"],
        processor,
        max_samples=CONFIG["max_tune_val_samples"],
    )
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    val_loss = run_validation_loss(model, val_loader, device)
    print(f"Post-training val-split loss (sanity check, not the test metric): {val_loss:.4f}")

    return run_ckpt_dir

In [ ]:
# ----------------------------------------------------------------------
# 3. INFERENCE
# ----------------------------------------------------------------------
def generate_captions(model, processor, image_paths, device, batch_size=16):
    model.eval()
    results = {}
    with torch.no_grad():
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i + batch_size]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
            out_ids = model.generate(**inputs, max_length=CONFIG["max_caption_len"], num_beams=4)
            captions = processor.batch_decode(out_ids, skip_special_tokens=True)
            for p, c in zip(batch_paths, captions):
                results[p] = c.strip()
    return results

In [ ]:
def get_output_dir(checkpoint_path):
    """
    Maps a checkpoint path to its own output folder, e.g.
      ./checkpoints/blip_1epoch/epoch0  -> outputs/blip_1epoch_epoch0/
      ./checkpoints/blip_epoch2         -> outputs/blip_epoch2/
      None (pretrained baseline)        -> outputs/pretrained_baseline/
    """
    if checkpoint_path is None:
        tag = "blip-base-coco"
    else:
        p = Path(checkpoint_path)
        try:
            tag = "_".join(p.relative_to(CONFIG["checkpoint_dir"]).parts)
        except ValueError:
            tag = p.name
    out_dir = Path(CONFIG["output_dir"]) / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

In [ ]:
def evaluate(checkpoint_path="./checkpoints/blip_1epoch/epoch0"):
    """
    Computes BLEU-4 (smoothed), METEOR, ROUGE-L via nltk/rouge-score
    (lightweight, no Java dependency), and CIDEr via pycocoevalcap
    (needs Java + the original COCO eval toolkit format).
 
    Evaluated ONLY on captions_val2017.json (CONFIG['test_ann']) — the
    true held-out test set, never seen during training or tuning.
    """
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
 
    device = CONFIG["device"]
    print(f"Device: {device}")
 
    model_path = checkpoint_path or CONFIG["model_name"]
    print(f"Loading model from: {model_path}")
 
    processor = BlipProcessor.from_pretrained(model_path)
    model = BlipForConditionalGeneration.from_pretrained(model_path).to(device)
    print("Model loaded.")
 
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["test_ann"]) as f:
        coco = json.load(f)
 
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    refs_by_image = {}
    for a in coco["annotations"]:
        refs_by_image.setdefault(a["image_id"], []).append(a["caption"].strip())
 
    image_ids = list(refs_by_image.keys())[: CONFIG["max_test_samples"]]
    image_paths = [str(root / CONFIG["test_img_dir"] / id_to_file[i]) for i in image_ids]
    print(f"Evaluating on {len(image_paths)} held-out TEST images (val2017)...")
 
    preds = generate_captions(model, processor, image_paths, device)
    print("Caption generation done. Computing metrics...")
 
    smoothie = SmoothingFunction().method4
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
 
    bleu_scores, meteor_scores, rouge_scores = [], [], []
    coco_gts, coco_res = {}, {}  # for optional CIDEr via pycocoevalcap
 
    for img_id, path in zip(image_ids, image_paths):
        hyp = preds[path]
        refs = refs_by_image[img_id]
        ref_tokens = [r.split() for r in refs]
 
        bleu_scores.append(sentence_bleu(ref_tokens, hyp.split(), smoothing_function=smoothie))
        meteor_scores.append(meteor_score(ref_tokens, hyp.split()))
        rouge_scores.append(
            max(scorer.score(r, hyp)["rougeL"].fmeasure for r in refs)
        )
 
        coco_gts[img_id] = refs
        coco_res[img_id] = [hyp]
 
    print(f"BLEU-4 (avg, smoothed): {sum(bleu_scores) / len(bleu_scores):.4f}")
    print(f"METEOR (avg):           {sum(meteor_scores) / len(meteor_scores):.4f}")
    print(f"ROUGE-L (avg F1):       {sum(rouge_scores) / len(rouge_scores):.4f}")
 
    # CIDEr — requires pycocoevalcap (`pip install pycocoevalcap`) and Java.
    try:
        from pycocoevalcap.cider.cider import Cider
        cider_scorer = Cider()
        cider_score, _ = cider_scorer.compute_score(coco_gts, coco_res)
        print(f"CIDEr:                  {cider_score:.4f}")
    except ImportError:
        print("pycocoevalcap not installed — skipping CIDEr. "
              "Install with: pip install pycocoevalcap")
    metrics = {
        "BLEU-4": sum(bleu_scores) / len(bleu_scores),
        "METEOR": sum(meteor_scores) / len(meteor_scores),
        "ROUGE-L": sum(rouge_scores) / len(rouge_scores),
        "CIDEr": cider_score
    }
    out_dir = get_output_dir(checkpoint_path)
    with open(out_dir / "predictions.json", "w") as f:
        json.dump(preds, f, indent=2)
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Predictions + metrics saved to {out_dir}")

In [ ]:
def qualitative_analysis(checkpoint_path=None, n_samples=8):
    """
    Prints/saves a grid of (image, generated caption, ground-truth captions)
    for manual inspection of fluency, relevance, and diversity. Samples
    come from the held-out test set (val2017), same as evaluate().
    """
    import matplotlib.pyplot as plt
 
    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    processor = BlipProcessor.from_pretrained(model_path)
    model = BlipForConditionalGeneration.from_pretrained(model_path).to(device)
 
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["test_ann"]) as f:
        coco = json.load(f)
 
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    refs_by_image = {}
    for a in coco["annotations"]:
        refs_by_image.setdefault(a["image_id"], []).append(a["caption"].strip())
 
    sample_ids = random.Random(CONFIG["seed"]).sample(list(refs_by_image.keys()), n_samples)
    image_paths = [str(root / CONFIG["test_img_dir"] / id_to_file[i]) for i in sample_ids]
    preds = generate_captions(model, processor, image_paths, device)
 
    fig, axes = plt.subplots(2, n_samples // 2, figsize=(4 * n_samples // 2, 8))
    for ax, img_id, path in zip(axes.flat, sample_ids, image_paths):
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.axis("off")
        gt = refs_by_image[img_id][0]
        ax.set_title(f"Pred: {preds[path]}\nGT: {gt}", fontsize=8, wrap=True)
 
    out_dir = get_output_dir(checkpoint_path)
    fig.tight_layout()
    fig.savefig(out_dir / "qualitative_grid.png", dpi=150)
    print(f"Saved qualitative grid to {out_dir / 'qualitative_grid.png'}")
 
 
# ----------------------------------------------------------------------
# 6. ZERO-SHOT / OUT-OF-DOMAIN EVALUATION (e.g. NoCaps, or your own images)
# ----------------------------------------------------------------------
def zero_shot(image_dir, checkpoint_path=None, cols=3):
    import matplotlib.pyplot as plt
    import textwrap

    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    processor = BlipProcessor.from_pretrained(model_path)
    model = BlipForConditionalGeneration.from_pretrained(model_path).to(device)

    image_paths = [
        str(p) for p in sorted(Path(image_dir).glob("*"))
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    preds = generate_captions(model, processor, image_paths, device)
    for p, c in preds.items():
        print(f"{Path(p).name}: {c}")

    # ---- image + caption grid ----
    n = len(image_paths)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = axes.reshape(-1) if n > 1 else [axes]
    for ax in axes:
        ax.axis("off")
    for ax, p in zip(axes, image_paths):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(textwrap.fill(preds[p], 40), fontsize=9)

    out_dir = get_output_dir(checkpoint_path)          # use Path(CONFIG["output_dir"]) if you haven't added get_output_dir to BLIP
    fig.tight_layout()
    fig.savefig(out_dir / "zero_shot_grid.png", dpi=150)
    plt.show()

    with open(out_dir / "zero_shot_predictions.json", "w") as f:
        json.dump(preds, f, indent=2)
    print(f"Saved to {out_dir}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F


def visualize_attention(checkpoint_path=None, image_paths=None, n_samples=4, layer=-1):
    """
    Overlays BLIP's vision-encoder attention (from the chosen ViT layer,
    averaged across heads) on top of the input image, showing which
    patches the model attended to most when encoding the image.

    image_paths: list of image file paths to visualize. If None, pulls
    `n_samples` random images from the held-out test set (val2017).
    layer: which ViT encoder layer to visualize (-1 = last layer).
    """
    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    processor = BlipProcessor.from_pretrained(model_path)
    model = BlipForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()

    if image_paths is None:
        root = Path(CONFIG["data_root"])
        with open(root / CONFIG["test_ann"]) as f:
            coco = json.load(f)
        id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
        sample_ids = random.Random(CONFIG["seed"]).sample(list(id_to_file.keys()), n_samples)
        image_paths = [str(root / CONFIG["test_img_dir"] / id_to_file[i]) for i in sample_ids]

    n = len(image_paths)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)

    for col, path in enumerate(image_paths):
        image = Image.open(path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)

        with torch.no_grad():
            vision_outputs = model.vision_model(
                pixel_values=inputs["pixel_values"],
                output_attentions=True,
            )
            # Also generate a caption so we can show it as context.
            out_ids = model.generate(**inputs, max_length=CONFIG["max_caption_len"], num_beams=4)
            caption = processor.batch_decode(out_ids, skip_special_tokens=True)[0].strip()

        # attentions: tuple of (num_layers), each (batch, heads, tokens, tokens)
        attn = vision_outputs.attentions[layer][0]       # (heads, tokens, tokens)
        attn = attn.mean(dim=0)                           # average over heads -> (tokens, tokens)

        # Attention FROM the [CLS] token (index 0) TO every patch token.
        cls_attn = attn[0, 1:]                             # drop self-attention to CLS, keep patches
        n_patches = cls_attn.shape[0]
        grid_size = int(n_patches ** 0.5)                  # e.g. 24x24 for BLIP-base
        cls_attn = cls_attn.reshape(grid_size, grid_size)

        # Upsample the low-res attention grid to the image size for overlay.
        cls_attn = cls_attn.unsqueeze(0).unsqueeze(0)       # (1,1,g,g)
        cls_attn = F.interpolate(cls_attn, size=image.size[::-1], mode="bilinear", align_corners=False)
        cls_attn = cls_attn.squeeze().cpu().numpy()
        cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)

        axes[0, col].imshow(image)
        axes[0, col].axis("off")
        axes[0, col].set_title(caption, fontsize=8, wrap=True)

        axes[1, col].imshow(image)
        axes[1, col].imshow(cls_attn, cmap="jet", alpha=0.5)
        axes[1, col].axis("off")
        axes[1, col].set_title(f"Layer {layer} attention", fontsize=8)

    out_dir = get_output_dir(checkpoint_path)
    fig.tight_layout()
    fig.savefig(out_dir / "attention_grid.png", dpi=150)
    print(f"Saved attention grid to {out_dir / 'attention_grid.png'}")

In [ ]:
import textwrap

def prompted_captioning(checkpoint_path=None, prompts=None, n_samples=4):
    """
    Prompt-conditioned captioning: the model must continue a text prefix
    (e.g. "a photography of") instead of generating a caption from scratch.
    Compares unprompted vs prompted captions on the same fixed test images.
    """
    import matplotlib.pyplot as plt

    prompts = prompts or ["a photography of", "a picture of", "a close up of"]
    device = CONFIG["device"]
    model_path = checkpoint_path or CONFIG["model_name"]
    processor = BlipProcessor.from_pretrained(model_path)
    model = BlipForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()

    # Same fixed sampling as qualitative_analysis / visualize_attention,
    # so every grid in your report shows the same images.
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["test_ann"]) as f:
        coco = json.load(f)
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    sample_ids = random.Random(CONFIG["seed"]).sample(list(id_to_file.keys()), n_samples)
    image_paths = [str(root / CONFIG["test_img_dir"] / id_to_file[i]) for i in sample_ids]
    images = [Image.open(p).convert("RGB") for p in image_paths]

    def run(prompt=None):
        if prompt is None:
            inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
        else:
            inputs = processor(images=images, text=[prompt] * len(images),
                               return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_length=CONFIG["max_caption_len"], num_beams=4)
        return [c.strip() for c in processor.batch_decode(out_ids, skip_special_tokens=True)]

    results = {"unprompted": run(None)}
    for p in prompts:
        results[p] = run(p)

    # ---- plot: image on top, all captions underneath ----
    fig, axes = plt.subplots(2, n_samples, figsize=(4.5 * n_samples, 9),
                             gridspec_kw={"height_ratios": [1, 1.2]})
    if n_samples == 1:
        axes = axes.reshape(2, 1)
    for col, img in enumerate(images):
        axes[0, col].imshow(img)
        axes[0, col].axis("off")
        lines = []
        for name, caps in results.items():
            label = "no prompt" if name == "unprompted" else f'"{name}"'
            lines.append(f"[{label}]\n" + textwrap.fill(caps[col], 40))
        axes[1, col].text(0, 1, "\n\n".join(lines), va="top", fontsize=8)
        axes[1, col].axis("off")

    out_dir = get_output_dir(checkpoint_path)
    fig.tight_layout()
    fig.savefig(out_dir / "prompted_captions.png", dpi=150)
    with open(out_dir / "prompted_captions.json", "w") as f:
        json.dump({"images": image_paths, "captions": results}, f, indent=2)
    print(f"Saved prompted captions to {out_dir}")
    return results

In [ ]:
prepare_data()

In [ ]:
tune_hyperparameters(n_trials=16)

In [ ]:
evaluate()

In [ ]:
qualitative_analysis()

In [ ]:
visualize_attention(n_samples=4, layer=-1)

In [ ]:
prompted_captioning()

In [ ]:
zero_shot(image_dir="./image_dir")

In [ ]:
train(run_name="blip_1epoch", num_epochs=1)

In [ ]:
evaluate("./checkpoints/blip_1epoch/epoch0")

In [ ]:
qualitative_analysis(checkpoint_path="./checkpoints/blip_1epoch/epoch0")

In [ ]:
visualize_attention(checkpoint_path="./checkpoints/blip_1epoch/epoch0", n_samples=4, layer=-1)

In [ ]:
prompted_captioning("./checkpoints/blip_1epoch/epoch0")

In [ ]:
zero_shot(image_dir="./image_dir", checkpoint_path="./checkpoints/blip_1epoch/epoch0")

In [ ]:
train(run_name="blip_3epoch", num_epochs=3)

In [ ]:
evaluate("./checkpoints/blip_3epoch/epoch2")

In [ ]:
qualitative_analysis("./checkpoints/blip_3epoch/epoch2")

In [ ]:
visualize_attention(checkpoint_path="./checkpoints/blip_3epoch/epoch2", n_samples=4, layer=-1)

In [ ]:
prompted_captioning(checkpoint_path="./checkpoints/blip_3epoch/epoch2", prompts=["a photography of", "a close up of", "two people"], n_samples=4)

In [ ]:
zero_shot(image_dir="./image_dir", checkpoint_path="./checkpoints/blip_1epoch/epoch0")